# Track 1 · 01 Prepare — put the training material on Drive

**One job.** Write the two things Track 1 rung-1 training needs, and nothing else:

1. the EEG for a small, explicit set of recordings, and
2. the frozen DINOv2 target table, one 1536-d vector per stimulus image.

It does not epoch, does not run REVE, and does not train. Those are the next steps.

This notebook is self-contained. It clones nothing and imports nothing from the repo, so
uploading this one file to Colab is enough to run it.

---

## The target space is a fixed specification

Track 1 scores cosine retrieval in a frozen embedding space. A head trained against a
slightly different space still fits, still reports a plausible training loss, and then
retrieves badly. There is no error message for getting this wrong, so the settings below
are copied from the competition's own config rather than chosen:

```yaml
target:
  name: HuggingFaceImage
  model_name: facebook/dinov2-giant
  layers: 0.6667
  token_aggregation: mean
  imsize: 518
brain_model_output_size: 1536
```

Source: `neuralbench/tasks/eeg/image/config.yaml`, neuralbench 0.3.1, retrieved 2026-09-24.

What those values resolve to, traced through `neuralset/extractors/image.py` and
`neuralset/extractors/hf.py`:

| setting | what actually happens |
|---|---|
| `layers: 0.6667` | `int(0.6667 x 40) = 26`, so **hidden state 26 of 41**. Truncation, not rounding. |
| `token_aggregation: mean` | mean over all 257 tokens, CLS included |
| `imsize: 518` | resize shorter side to 518, then the HF processor resizes to 256 and centre-crops 224 |

The `imsize` step is nearly cancelled by the processor, and neuralset warns as much. It is
kept because it is not *entirely* cancelled: resampling twice is not the same as resampling
once, and the reference table was built with it.

The embedding code below was checked against neuralset's own extractor on the same images
and agreed to `max |diff| = 0.0`. That check ran on `dinov2-base`, since the code path is
identical and dimension-agnostic but the download is 350 MB instead of 4.5 GB.

## What it costs

With the defaults below, 12 subjects contributing one training run and one test run each:

| | |
|---|---|
| EEG on Drive | ~125 MB (24 recordings) |
| embedding table | ~77 MB (~12,500 images x 1536 float32) |
| images retained | none, they are streamed and discarded |
| **Drive total** | **~200 MB** |

The embedding pass is resumable: progress is written in 512-image shards, so a dropped
session costs one shard rather than the whole pass. Re-running from the top is cheap. The
EEG download reuses whatever is already on Drive, and the embedding pass skips images it
has already done. The selection is deterministic, so a re-run always picks the same
recordings.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'   # tracks go in ROOT/Track1, ROOT/Track2, ...

N_SUBJECTS = 12            # subjects contributing recordings
RUNS_PER_SUBJECT = 1       # training runs each, and the same number of test runs

MODEL_NAME = 'facebook/dinov2-giant'   # the scored target space; do not change to fit
DEPTH = 0.6667                         # layers, from the track config
IMSIZE = 518
EMBED_BATCH = 16           # images per forward pass
FETCH_WORKERS = 16         # concurrent downloads
SHARD = 512                # embeddings per resumable shard

print(f'root       {ROOT}')
print(f'subjects   {N_SUBJECTS} x ({RUNS_PER_SUBJECT} train + {RUNS_PER_SUBJECT} test) runs')
print(f'target     {MODEL_NAME} @ depth {DEPTH}, mean tokens, imsize {IMSIZE}')

In [ ]:
%pip install -q 'eegdash>=0.9.1' mne transformers torchvision pillow requests

import importlib
for mod in ('eegdash', 'mne', 'transformers', 'torchvision', 'PIL', 'requests', 'torch'):
    m = importlib.import_module(mod)
    print(f'{mod:14} {getattr(m, "__version__", "ok")}')

### Helpers

Everything Track 1 needs, inline. The repo carries fuller versions of these in `src/`, but
keeping them here is what lets this notebook run as a single uploaded file.

In [ ]:
from pathlib import Path

DATASET = 'nm000134'              # Alljoined-1.6M on NeMAR
SLUG = 'Track1'                   # folder under ROOT
EXCLUDE_SESSIONS = ('02old',)     # superseded recordings
EXPECT_SFREQ, EXPECT_NCHANS = 256.0, 32

# Alljoined splits its stimuli by run: runs 01-04 present the 200 fixed test images,
# runs 05-19 present training images, and the two sets do not intersect. The run number
# *is* the cross-stimulus split.
TRAIN_RUNS = tuple(f'{i:02d}' for i in range(5, 20))
TEST_RUNS = tuple(f'{i:02d}' for i in range(1, 5))

BYTES_PER_SAMPLE = 4              # float32


def paths(root):
    base = Path(root) / SLUG
    out = {'base': base, 'raw': base / 'raw', 'prepared': base / 'prepared',
           'weights': base / 'weights', 'submissions': base / 'submissions'}
    for v in out.values():
        v.mkdir(parents=True, exist_ok=True)
    out['manifest'] = base / 'manifest.json'
    return out


def estimate_mb(record):
    """Uncompressed size of one recording, from metadata alone. The EDFs are ~half this."""
    return (record.get('ntimes') or 0) * (record.get('nchans') or 0) * BYTES_PER_SAMPLE / 1e6


def targets_mb(n_stimuli, dim=1536):
    return n_stimuli * dim * BYTES_PER_SAMPLE / 1e6


def select_run_split(records, n_subjects=12, runs_per_subject=1,
                     exclude_sessions=EXCLUDE_SESSIONS):
    """Split by run id, taking both sides of the split from one session per subject.

    Both sides come from the same session on purpose. Track 1's shift is cross-stimulus,
    so holding subject and session fixed keeps a local score about images rather than
    about session drift. Which session a subject contributes rotates with its index, so a
    one-run-per-subject selection is not silently all ses-01.

    Deterministic, so re-running picks the same recordings and a resumed run still matches.
    """
    exclude = set(exclude_sessions)
    grouped = {}
    for r in records:
        if r.get('_has_missing_files') or r.get('session') in exclude:
            continue
        grouped.setdefault(r['subject'], {}).setdefault(r['session'], []).append(r)

    train, test = [], []
    for i, subject in enumerate(sorted(grouped)[:n_subjects]):
        sessions = sorted(grouped[subject])
        order = sessions[i % len(sessions):] + sessions[:i % len(sessions)]
        for session in order:
            pool = grouped[subject][session]
            tr = sorted((r for r in pool if r['run'] in TRAIN_RUNS), key=lambda r: r['run'])
            te = sorted((r for r in pool if r['run'] in TEST_RUNS), key=lambda r: r['run'])
            if len(tr) < runs_per_subject or len(te) < runs_per_subject:
                continue
            tr = tr[i % len(tr):] + tr[:i % len(tr)]   # vary which images each subject brings
            train += tr[:runs_per_subject]
            test += te[:runs_per_subject]
            break
    return train, test


def report(title, checks):
    """checks: list of (name, ok, detail)."""
    print(title)
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    print(f'  -> {sum(c[1] for c in checks)} ok, {sum(not c[1] for c in checks)} failed')


print('helpers defined')

In [ ]:
import os

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
P = paths(ROOT)

# DINOv2-giant is 4.5 GB and is needed once. Keeping it on ephemeral /content costs a
# re-download on a new session and saves the Drive space for data that must persist.
os.environ['HF_HOME'] = '/content/hf'

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'dataset    {DATASET}')
print(f'raw        {P["raw"]}')
print(f'prepared   {P["prepared"]}')
print(f'device     {DEVICE}')
if DEVICE == 'cpu':
    print('\nNo GPU. dinov2-giant on CPU will take hours for this many images.')
    print('Switch the runtime to a GPU, or set MODEL_NAME to facebook/dinov2-base for a dry run.')

## Step 1 — choose the recordings

Selection is by run, not by byte budget, because the run number is the train/test split.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import mne
mne.set_log_level('ERROR')
from eegdash import EEGDash, EEGDashDataset

records = [r for r in EEGDash().find({'dataset': DATASET})
           if r['bids_relpath'].startswith('sub-')]
print(f'catalogue: {len(records)} records, {len({r["subject"] for r in records})} subjects')

train_recs, test_recs = select_run_split(
    records, n_subjects=N_SUBJECTS, runs_per_subject=RUNS_PER_SUBJECT)

for a, b in zip(train_recs, test_recs):
    print(f'  sub-{a["subject"]}  ses-{a["session"]}   '
          f'train run-{a["run"]}   test run-{b["run"]}')

selected = train_recs + test_recs
split_of = {r['bids_relpath']: s for recs, s in
            ((train_recs, 'train'), (test_recs, 'test')) for r in recs}

sfreqs = {r.get('sampling_frequency') for r in selected}
nchans = {r.get('nchans') for r in selected}
report('selection:', [
    ('records found', len(selected) == 2 * N_SUBJECTS * RUNS_PER_SUBJECT,
     f'{len(selected)} records, ~{sum(estimate_mb(r) for r in selected):.0f} MB uncompressed'),
    ('sampling rate', sfreqs == {EXPECT_SFREQ}, f'{sfreqs} Hz'),
    ('channel count', nchans == {EXPECT_NCHANS}, f'{nchans}'),
    ('subjects', len({r['subject'] for r in selected}) == N_SUBJECTS,
     f'{len({r["subject"] for r in selected})} distinct'),
    ('excluded sessions absent',
     not ({r['session'] for r in selected} & set(EXCLUDE_SESSIONS)), f'{EXCLUDE_SESSIONS}'),
])

## Step 2 — pull the EEG

Already-downloaded recordings are reused, so re-running this is free.

In [ ]:
import time

os.environ['EEGDASH_CACHE_DIR'] = str(P['raw'])
ds = EEGDashDataset(records=selected, cache_dir=P['raw'])

t0 = time.time()
fetched, failed = [], []
for i, d in enumerate(ds.datasets, 1):
    try:
        _ = d.raw                      # touching .raw is what downloads
        fetched.append(d)
    except Exception as e:
        failed.append((i, f'{type(e).__name__}: {e}'))
        print(f'  record {i} failed: {type(e).__name__}: {str(e)[:90]}')
print(f'{len(fetched)}/{len(selected)} recordings in {time.time() - t0:.0f}s')

on_disk_mb = sum(f.stat().st_size for f in P['raw'].rglob('*') if f.is_file()) / 1e6
print(f'{on_disk_mb:.0f} MB on Drive')

if fetched:
    raw = fetched[0].raw
    has_montage = raw.get_montage() is not None
    report('first recording:', [
        ('sampling rate', raw.info['sfreq'] == EXPECT_SFREQ, f'{raw.info["sfreq"]} Hz'),
        ('channel count', len(raw.ch_names) == EXPECT_NCHANS, f'{len(raw.ch_names)}'),
        ('annotations', len(raw.annotations) > 0, f'{len(raw.annotations)} events'),
        ('montage present', has_montage,
         'none, but REVE reads positions from channel names, not from the data'
         if not has_montage else 'yes'),
    ])

## Step 3 — read the stimulus lists

EEGDash caches each recording's BIDS `_events.tsv` next to the EDF, so the image shown at
every onset can be read directly.

The key is `stim_file`, not `value`. The dataset's own `events.json` records that `value` is
zero-padded to five digits in eleven sessions and is a placeholder for oddball rows in seven
more. `stim_file` names the JPG in every row that has one.

In [ ]:
import pandas as pd

STIM_KINDS = {'train': 'stim_train', 'test': 'stim_test'}
events, images = {}, {'train': set(), 'test': set()}

for rec in selected:
    rel = rec['bids_relpath']
    split = split_of[rel]
    tsv = P['raw'] / DATASET / rel.replace('_eeg.edf', '_events.tsv')
    if not tsv.exists():
        print(f'  no events.tsv for {rel}')
        continue
    df = pd.read_csv(tsv, sep='\t')
    df = df[df.trial_type == STIM_KINDS[split]].dropna(subset=['stim_file'])
    events[rel] = df[['onset', 'sample', 'stim_file']].reset_index(drop=True)
    images[split] |= set(df.stim_file)

n_ev = {s: sum(len(v) for k, v in events.items() if split_of[k] == s) for s in ('train', 'test')}
print(f'train   {n_ev["train"]:6,} events   {len(images["train"]):6,} unique images')
print(f'test    {n_ev["test"]:6,} events   {len(images["test"]):6,} unique images')

overlap = images['train'] & images['test']
print(f'\ntrain/test image overlap: {len(overlap)}   (must be 0 for a cross-stimulus split)')
assert not overlap, f'{len(overlap)} images appear in both splits'

all_images = sorted(images['train'] | images['test'])
print(f'to embed: {len(all_images):,} images  ->  {targets_mb(len(all_images)):.0f} MB table')
print(f'local gallery for scoring: {len(images["test"])} images, '
      f'chance top-5 {5 / max(len(images["test"]), 1):.2%}')

## Step 4 — build the target table

The model is loaded once and the images are streamed past it. Nothing is kept but the
1536-d vector.

The self-test is the guard against a silently wrong target space. It asserts the layer
index and the output dimension, which are the two things that would otherwise be wrong
without complaint. It runs before the long pass, so a mistake costs seconds.

In [ ]:
import io, requests, numpy as np, transformers
from PIL import Image as PILImage
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModel

STIM_URL = f'https://data.nemar.org/{DATASET}/v1.0.3/stimuli/{{}}'

_tf = transforms.Compose([transforms.Resize(IMSIZE), transforms.ToTensor()])
_proc = AutoImageProcessor.from_pretrained(MODEL_NAME, do_rescale=False)
_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

_session = requests.Session()
# The default pool holds 10 connections; with more workers than that, requests
# discards and re-opens sockets for every batch.
_session.mount('https://', requests.adapters.HTTPAdapter(
    pool_connections=FETCH_WORKERS, pool_maxsize=FETCH_WORKERS, max_retries=2))


def fetch(name):
    """Stream one stimulus. The nemar URL 302-redirects to presigned S3."""
    r = _session.get(STIM_URL.format(name), timeout=60)
    r.raise_for_status()
    return PILImage.open(io.BytesIO(r.content)).convert('RGB')


def _safe_fetch(name):
    """Fetch, or None. Keeps one bad URL from ending a 12,000-image pass."""
    try:
        return fetch(name)
    except Exception:
        return None


def embed(pil_images):
    """The target space, exactly as neuralbench defines it. Returns (B, 1536) float32."""
    tens = [_tf(im).float() for im in pil_images]
    inputs = _proc(images=tens, return_tensors='pt')
    pv = inputs['pixel_values']
    if pv.isnan().any():                 # uniform images produce NaNs; neuralset zeroes them
        pv[pv.isnan()] = 0
        inputs['pixel_values'] = pv.float()
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.inference_mode():
        states = _model(**inputs, output_hidden_states=True).hidden_states
    if states is None:
        raise RuntimeError(f'{MODEL_NAME} returned no hidden states on transformers '
                           f'{transformers.__version__}; hidden-state collection regressed '
                           'for some encoders in transformers 5')
    stacked = torch.cat([s.unsqueeze(1) for s in states], dim=1)   # (B, layers, tokens, F)
    n_layers = stacked.shape[1]
    idx = int(DEPTH * (n_layers - 1))    # truncation, matching neuralset
    return stacked[:, idx].mean(dim=1).float().cpu().numpy(), idx, n_layers


probe = [fetch(n) for n in all_images[:4]]
vecs, layer_idx, n_layers = embed(probe)
print(f'{MODEL_NAME}: {n_layers} hidden states, taking index {layer_idx}')
print(f'embedding {vecs.shape}, norms {np.linalg.norm(vecs, axis=1).round(2)}')

if MODEL_NAME == 'facebook/dinov2-giant':
    assert (n_layers, layer_idx) == (41, 26), (n_layers, layer_idx)
    assert vecs.shape[1] == 1536, vecs.shape
    print('matches the scored target space')
else:
    print(f'DRY RUN on {MODEL_NAME}: this table is NOT the scored target space')
del probe, vecs

In [ ]:
from concurrent.futures import ThreadPoolExecutor

parts = P['prepared'] / 'embedding_parts'
parts.mkdir(parents=True, exist_ok=True)

done = set()
for f in sorted(parts.glob('part_*.npz')):
    done |= set(np.load(f)['keys'].tolist())
todo = [n for n in all_images if n not in done]
print(f'{len(done):,} already embedded, {len(todo):,} to go')

pool = ThreadPoolExecutor(FETCH_WORKERS)
buf_keys, buf_vecs, failures = [], [], []
shard_no = len(list(parts.glob('part_*.npz')))
t0 = time.time()


def flush():
    global buf_keys, buf_vecs, shard_no
    if not buf_keys:
        return
    np.savez(parts / f'part_{shard_no:05d}.npz',
             keys=np.array(buf_keys), emb=np.stack(buf_vecs).astype('float32'))
    shard_no += 1
    buf_keys, buf_vecs = [], []


for start in range(0, len(todo), EMBED_BATCH):
    names = todo[start:start + EMBED_BATCH]
    got, imgs = [], []
    for name, im in pool.map(lambda n: (n, _safe_fetch(n)), names):
        if im is None:
            failures.append(name)
        else:
            got.append(name)
            imgs.append(im)
    if imgs:
        vecs, _, _ = embed(imgs)
        buf_keys += got
        buf_vecs += list(vecs)
    if len(buf_keys) >= SHARD:
        flush()
    n_seen = start + len(names)
    if n_seen % (EMBED_BATCH * 20) == 0 or n_seen >= len(todo):
        rate = n_seen / max(time.time() - t0, 1e-9)
        print(f'  {n_seen:6,}/{len(todo):,}  {rate:5.1f} img/s  '
              f'eta {(len(todo) - n_seen) / max(rate, 1e-9) / 60:5.1f} min  '
              f'{len(failures)} failed')

flush()
print(f'\ndone in {(time.time() - t0) / 60:.1f} min, {len(failures)} failures')

In [ ]:
# Retry anything that failed, once, serially. A failure here is usually a transient
# 5xx from the redirect, not a missing image.
if failures:
    print(f'retrying {len(failures)}')
    retry_keys, retry_vecs, still = [], [], []
    for name in failures:
        im = _safe_fetch(name)
        if im is None:
            still.append(name)
            continue
        v, _, _ = embed([im])
        retry_keys.append(name)
        retry_vecs.append(v[0])
    if retry_keys:
        np.savez(parts / f'part_{shard_no:05d}.npz',
                 keys=np.array(retry_keys), emb=np.stack(retry_vecs).astype('float32'))
        shard_no += 1
    failures = still
    print(f'{len(failures)} still failing: {failures[:10]}')
else:
    print('nothing to retry')

## Step 5 — consolidate and record what was written

In [ ]:
keys, mat = [], []
for f in sorted(parts.glob('part_*.npz')):
    z = np.load(f)
    keys += z['keys'].tolist()
    mat.append(z['emb'])

emb = np.concatenate(mat) if mat else np.zeros((0, 1536), 'float32')
order = {k: i for i, k in enumerate(keys)}          # last write wins on a duplicate
uniq = sorted(order)
emb = emb[[order[k] for k in uniq]]

table = P['prepared'] / 'image_embeddings.npz'
np.savez(table, keys=np.array(uniq), emb=emb, model=MODEL_NAME, depth=DEPTH,
         imsize=IMSIZE, token_aggregation='mean')
print(f'{table}  {emb.shape}  {table.stat().st_size / 1e6:.0f} MB')

missing = sorted(set(all_images) - set(uniq))
print(f'coverage: {len(uniq):,}/{len(all_images):,} images, {len(missing)} missing')
if missing:
    print(f'  missing: {missing[:10]}')

# Coverage says every image produced a row. It cannot say the rows are usable, so
# check the values too: a degenerate embedding would pass the count and poison the fit.
norms = np.linalg.norm(emb, axis=1)
report('table:', [
    ('shape', emb.shape[1] == (1536 if MODEL_NAME.endswith('giant') else emb.shape[1]),
     f'{emb.shape} {emb.dtype}'),
    ('no missing images', not missing, f'{len(missing)} missing'),
    ('finite values', bool(np.isfinite(emb).all()), 'no NaN or Inf'),
    ('distinct keys', len(uniq) == len(set(uniq)), f'{len(uniq):,} keys'),
    ('no zero rows', bool((norms > 0).all()), f'{int((norms == 0).sum())} all-zero'),
    ('norms', bool(norms.max() / max(norms.min(), 1e-9) < 10),
     f'min {norms.min():.2f}  median {np.median(norms):.2f}  max {norms.max():.2f}'),
])

In [ ]:
import json

meta = {
    'dataset': DATASET,
    'target': {'model': MODEL_NAME, 'layers': DEPTH, 'token_aggregation': 'mean',
               'imsize': IMSIZE, 'dim': int(emb.shape[1]),
               'source': 'neuralbench/tasks/eeg/image/config.yaml (neuralbench 0.3.1)'},
    'epoch': {'start_s': -0.2, 'duration_s': 1.0, 'baseline_s': [0.0, 0.2],
              'note': 'from the same config; ISI is ~200 ms so epochs overlap heavily'},
    'table': {'path': str(table), 'images': len(uniq), 'missing': missing},
    'recordings': {rel: {'split': split_of[rel],
                         'subject': rel.split('/')[0].removeprefix('sub-'),
                         'events': len(df),
                         'images': sorted(set(df.stim_file))}
                   for rel, df in events.items()},
    'counts': {'train_events': n_ev['train'], 'test_events': n_ev['test'],
               'train_images': len(images['train']), 'test_images': len(images['test'])},
}
(P['prepared'] / 'prepare_manifest.json').write_text(json.dumps(meta, indent=2))
print(f'manifest -> {P["prepared"] / "prepare_manifest.json"}')

used = sum(f.stat().st_size for f in P['base'].rglob('*') if f.is_file()) / 1e6
print(f'\nDrive used by {SLUG}: {used:.0f} MB')
print(f'  EEG        {on_disk_mb:6.0f} MB')
print(f'  embeddings {table.stat().st_size / 1e6:6.0f} MB')
print('\nNext: epoch the EEG at -0.2 to 0.8 s, run REVE over the windows, fit the ridge head.')
print('Note: no montage is set on these recordings. REVE does not need one: it looks')
print('up electrode positions by channel name in its own bank. Matters only for an')
print('encoder that reads positions out of the data.')